# SpaceX Falcon 9 First Stage Landing Prediction

## Lab 6: Launch Sites Location Analysis with Folium

The success of a launch may depend on many factors, such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors, and hopefully we can discover some of them by analyzing the existing launch site locations with interactive maps built with Folium.

## Objectives

- Mark all launch sites on a map
- Mark the success/failed launches for each site on the map
- Calculate the distances between a launch site and its proximities (coastline, highway, railway, city)

After completing these tasks, we should be able to find some geographical patterns about launch sites.

In [ ]:
!pip install folium -q
import folium
import pandas as pd
from folium.plugins import MarkerCluster
from folium.plugins import MousePosition
from folium.features import DivIcon

In [ ]:
URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
spacex_df = pd.read_csv(URL)
spacex_df.head()

### Task 1: Mark all launch sites on a map

First, we select the relevant columns and keep one row per launch site (using the first occurrence). Then we create a Folium `Map` object centered on the NASA Johnson Space Center in Houston, and add a `Circle` and a text `Marker` for each of the four launch sites: CCAFS LC-40, CCAFS SLC-40, KSC LC-39A and VAFB SLC-4E.

In [ ]:
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

In [ ]:
# Start location: NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

for index, site in launch_sites_df.iterrows():
    coordinate = [site['Lat'], site['Long']]
    circle = folium.Circle(coordinate, radius=1000, color='#d35400', fill=True).add_child(
        folium.Popup(site['Launch Site']))
    marker = folium.map.Marker(
        coordinate,
        icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
                     html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % site['Launch Site']))
    site_map.add_child(circle)
    site_map.add_child(marker)

site_map

The map shows that all launch sites are located very close to the coast — in Florida (Cape Canaveral and Kennedy Space Center) and in California (Vandenberg). Now we enhance the map by adding the launch outcome of every record.

### Task 2: Mark the success/failed launches for each site on the map

We create a `MarkerCluster` object. For each launch record, we add a marker whose color is **green** when the landing was successful (`class` = 1) and **red** when it failed (`class` = 0). Clustering keeps the map readable when many launches share the same site coordinates.

In [ ]:
marker_cluster = MarkerCluster()

# Apply a function to check the value of `class` column
# If class=1, marker_color value will be green
# If class=0, marker_color value will be red
def assign_marker_color(launch_outcome):
    if launch_outcome == 1:
        return 'green'
    else:
        return 'red'

spacex_df['marker_color'] = spacex_df['class'].apply(assign_marker_color)

site_map.add_child(marker_cluster)
for index, record in spacex_df.iterrows():
    marker = folium.Marker(
        [record['Lat'], record['Long']],
        icon=folium.Icon(color='white', icon_color=record['marker_color']),
        popup=record['Launch Site'])
    marker_cluster.add_child(marker)

site_map

From the color-labeled markers in the marker clusters, we can easily identify which launch sites have relatively high success rates: KSC LC-39A is mostly green, while the early CCAFS LC-40 records are mostly red.

### Task 3: Calculate the distances between a launch site and its proximities

We add a `MousePosition` plugin to read the coordinates of any point on the map, then use those coordinates to compute distances between a launch site and nearby points of interest. We draw a `PolyLine` between each pair and attach a marker showing the distance in km.

In [ ]:
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)
site_map.add_child(mouse_position)
site_map

In [ ]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    """Great-circle distance between two points on Earth (km)."""
    R = 6373.0
    lat1, lon1, lat2, lon2 = radians(lat1), radians(lon1), radians(lat2), radians(lon2)
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    distance = R * c
    return distance

In [ ]:
def add_distance_line(site_map, site_coordinate, target_coordinate, target_name):
    """Draw a PolyLine between the launch site and a point of interest, with a distance marker."""
    distance = calculate_distance(site_coordinate[0], site_coordinate[1],
                                  target_coordinate[0], target_coordinate[1])
    distance_marker = folium.Marker(
        target_coordinate,
        icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
                     html='<div style="font-size: 12; color:#d35400;"><b>%s %.2f KM</b></div>' % (target_name, distance)))
    site_map.add_child(distance_marker)
    lines = folium.PolyLine(locations=[site_coordinate, target_coordinate], weight=1)
    site_map.add_child(lines)
    return distance

In [ ]:
# Coordinates of CCAFS SLC-40 and its proximities (read with the MousePosition plugin)
launch_site_lat = 28.563197
launch_site_lon = -80.576820
launch_site_coordinate = [launch_site_lat, launch_site_lon]

coastline_coordinate = [28.56367, -80.57163]   # nearest coastline point
highway_coordinate = [28.56340, -80.57079]     # nearest highway (Samuel C Phillips Pkwy)
railway_coordinate = [28.57206, -80.58528]     # nearest railway
city_coordinate = [28.08363, -80.60811]        # city of Melbourne

d_coast = add_distance_line(site_map, launch_site_coordinate, coastline_coordinate, 'Coastline')
d_highway = add_distance_line(site_map, launch_site_coordinate, highway_coordinate, 'Highway')
d_rail = add_distance_line(site_map, launch_site_coordinate, railway_coordinate, 'Railway')
d_city = add_distance_line(site_map, launch_site_coordinate, city_coordinate, 'Melbourne')

print(f"Distance to coastline: {d_coast:.2f} km")
print(f"Distance to highway:   {d_highway:.2f} km")
print(f"Distance to railway:   {d_rail:.2f} km")
print(f"Distance to Melbourne: {d_city:.2f} km")

In [ ]:
site_map

## Findings

Using CCAFS SLC-40 as an example, the launch site is:

- Very close to the coastline (~0.86 km) — safe fallback area for failing rockets and drone-ship recovery
- Very close to the highway (~0.59 km) and the railway (~1.25 km) — easy transport of heavy rocket components
- Relatively far from the city of Melbourne (~51.43 km) — keeping populated areas safe

In general, launch sites are located near coasts, highways and railways for logistics and safety, while being kept at a safe distance from cities.